In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import LSTM, Dense, Dropout, Concatenate

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("final_sql_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (16400, 37)


,query,data,singlequotes,doublequotes,punctuations,1-linecmt,mulline-cmt,spaces,safekywrd,harmflkywrd,...,has_union,has_select,has_drop,has_insert,has_delete,has_update,has_or_1_equals_1,has_comment,has_sleep,has_semicolon
0,-4838' ) ) or 1 group by concat ( 0x...,0.0,1.0,0.0,21.0,0.0,0.0,42.0,8.0,2.0,...,0,1,0,0,0,0,0,1,0,0
1,1 or ( select * from ( select ( sleep ...,0.0,0.0,0.0,8.0,0.0,0.0,18.0,3.0,1.0,...,0,1,0,0,0,0,0,1,1,0
2,SELECT ALL column_name FROM table_name WHERE c...,1.0,0.0,0.0,1.0,0.0,0.0,8.0,2.0,0.0,...,0,1,0,0,0,0,0,0,0,1
3,"SELECT baseball, look, groundFROM busy WHER...",1.0,0.0,0.0,2.0,0.0,0.0,10.0,2.0,1.0,...,0,1,0,0,0,0,0,0,0,0
4,if ( 4194 = 4133 ) select 4194 else dro...,0.0,0.0,0.0,4.0,0.0,0.0,12.0,1.0,1.0,...,0,1,1,0,0,0,0,1,0,0


In [5]:
print(df['label'].value_counts())

label
1    10300
0     5300
2      800
Name: count, dtype: int64


In [6]:
# Separate classes
df_safe = df[df['label'] == 0]
df_injection = df[df['label'] == 1]

print("Before balancing:")
print(df['label'].value_counts())

# Downsample injection class
df_injection_sampled = df_injection.sample(len(df_safe), random_state=42)

# Combine
df_balanced = pd.concat([df_safe, df_injection_sampled])

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter balancing:")
print(df_balanced['label'].value_counts())

Before balancing:
label
1    10300
0     5300
2      800
Name: count, dtype: int64

After balancing:
label
1    5300
0    5300
Name: count, dtype: int64


In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Load your dataset
df = pd.read_csv("final_sql_dataset.csv")

# Separate features and label
X = df.drop(columns=["label", "query"])   # remove non-feature columns
y = df["label"]

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

# Get feature importance
importances = rf.feature_importances_

# Create DataFrame
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})

# Sort features
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

print(feature_importance_df)

              feature  importance
0                data    0.292008
19             digits    0.149933
3        punctuations    0.068271
32        has_comment    0.068128
24  count_parentheses    0.048029
21       query_length    0.046506
23       count_equals    0.045988
18          alphabets    0.038095
17          lang_cmds    0.032617
6              spaces    0.031397
11           operator    0.023312
22         num_quotes    0.021900
26         has_select    0.018110
20           spl_char    0.017136
25          has_union    0.015963
1        singlequotes    0.012352
2        doublequotes    0.010224
7           safekywrd    0.010153
8         harmflkywrd    0.009228
10          log_oprtr    0.008908
12              nulls    0.006296
34      has_semicolon    0.004501
16          ntwr_cmds    0.003571
33          has_sleep    0.003156
9         percentages    0.002961
15              roles    0.002753
27           has_drop    0.002688
13            hex-dec    0.001598
4           1-

In [8]:
top_n = 10
selected_features = feature_importance_df['feature'].head(top_n).tolist()

print(selected_features)

['data', 'digits', 'punctuations', 'has_comment', 'count_parentheses', 'query_length', 'count_equals', 'alphabets', 'lang_cmds', 'spaces']


In [9]:
import joblib
joblib.dump(selected_features, "selected_features.pkl")

['selected_features.pkl']

In [10]:
selected_features = [
    'digits','spl_char','query_length','num_quotes',
    'count_equals','count_parentheses','operator',
    'has_union','has_select','has_comment','safekywrd'
]

In [11]:
from sklearn.model_selection import train_test_split

X_text = df['query']
X_feat = df[selected_features]
y = df['label']

X_text_train, X_text_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X_text, X_feat, y, test_size=0.2, random_state=42
)

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_text_train)

In [13]:
X_train_seq = tokenizer.texts_to_sequences(X_text_train)
X_test_seq = tokenizer.texts_to_sequences(X_text_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_feat_train_scaled = scaler.fit_transform(X_feat_train)
X_feat_test_scaled = scaler.transform(X_feat_test)

In [15]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, LSTM
from tensorflow.keras.layers import Dense, Dropout, Concatenate

# Text branch
text_input = Input(shape=(100,))
x = Embedding(5000, 128)(text_input)
x = Conv1D(128, 5, activation='relu')(x)
x = MaxPooling1D(2)(x)
x = LSTM(64)(x)

# Feature branch
feat_input = Input(shape=(len(selected_features),))
y_feat = Dense(32, activation='relu')(feat_input)

# Combine
combined = Concatenate()([x, y_feat])

z = Dense(64, activation='relu')(combined)
z = Dropout(0.5)(z)
z = Dense(32, activation='relu')(z)

output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[text_input, feat_input], outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [17]:
model.fit(
    [X_train_pad, X_feat_train_scaled],
    y_train,
    validation_data=([X_test_pad, X_feat_test_scaled], y_test),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.8977 - loss: -52.4986 - val_accuracy: 0.9341 - val_loss: -248.0424
Epoch 2/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.9376 - loss: -1141.2646 - val_accuracy: 0.9277 - val_loss: -2510.0664
Epoch 3/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9395 - loss: -5166.0396 - val_accuracy: 0.9402 - val_loss: -8404.8740
Epoch 4/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 28s 68ms/step - accuracy: 0.9428 - loss: -13999.3516 - val_accuracy: 0.9460 - val_loss: -19418.7637
Epoch 5/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 24s 59ms/step - accuracy: 0.9405 - loss: -28415.1367 - val_accuracy: 0.9402 - val_loss: -35419.1680
Epoch 6/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.9346 - loss: -49125.2344 - val_accuracy: 0.9442 - val_loss: -59657.8438
Epoch 7/10
410/410 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.9420 - loss: -75740.2891 - val_accuracy: 0.9451 - val_loss: -89615.5312
Epoch 8/10
410/410 ━━━━━━━━━━

In [20]:
from sklearn.metrics import classification_report

y_pred = (model.predict([X_test_pad, X_feat_test_scaled]) > 0.5).astype(int)

print(classification_report(y_test, y_pred))

103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1046
           1       0.91      0.99      0.95      2074
           2       0.00      0.00      0.00       160

    accuracy                           0.94      3280
   macro avg       0.63      0.65      0.64      3280
weighted avg       0.89      0.94      0.91      3280



In [21]:
import joblib

model.save("sql_model.h5")
joblib.dump(tokenizer, "tokenizer.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(selected_features, "features.pkl")

['features.pkl']